# Constant-Current Discharge with the SPMe

Constant-current (CC) discharge simulation using `CellElectrothermal`, which wraps
PyBaMM's SPMe with a lumped thermal sub-model. PathSim integrates the coupled ODE system.


## Model

The **SPMe** extends the Single Particle Model (SPM) with electrolyte concentration
dynamics, significantly improving accuracy at moderate-to-high C-rates. Solid-phase
diffusion in each electrode follows:

$$
\frac{\partial c_{\mathrm{s},k}}{\partial t}
= \frac{D_{\mathrm{s},k}}{r^2}
  \frac{\partial}{\partial r}\!\left(r^2 \frac{\partial c_{\mathrm{s},k}}{\partial r}\right)
$$

The terminal voltage is determined by open-circuit potentials and Butler–Volmer overpotentials.
Cell temperature is tracked via PyBaMM's lumped thermal sub-model:

$$
m C_p \frac{dT}{dt} = \dot{Q} - UA\,(T - T_{\mathrm{amb}})
$$

> Brosa Planella et al., [arXiv:2203.16091](https://arxiv.org/abs/2203.16091) (2022).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Constant, Scope
from pathsim.solvers import ESDIRK43

from pathsim_batt import CellElectrothermal

## Single 1 C Discharge

Chen2020 is a 21700-format NMC/graphite cell with 5 Ah nominal capacity, so 1 C = 5 A.
`ESDIRK43` is used because the discretised SPMe ODE is stiff.


In [ ]:
C_nom  = 5.0    # Chen2020 nominal capacity [Ah]
T_amb0 = 298.15 # [K]

cell  = CellElectrothermal(initial_soc=1.0)
I_src = Constant(1.0 * C_nom)
T_src = Constant(T_amb0)
sco   = Scope(labels=["V", "T", "Q_heat", "SOC"])

sim = Simulation(
    blocks=[I_src, T_src, cell, sco],
    connections=[
        Connection(I_src,          cell["I"]),
        Connection(T_src,          cell["T_amb"]),
        Connection(cell["V"],      sco[0]),
        Connection(cell["T"],      sco[1]),
        Connection(cell["Q_heat"], sco[2]),
        Connection(cell["SOC"],    sco[3]),
    ],
    dt=10.0,
    Solver=ESDIRK43,
)

sim.run(3600.0)
t, [V, T, Q_heat, SOC] = sco.read()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)

axes[0].plot(t / 3600, V, color="steelblue")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_ylim(2.4, 4.3)
axes[0].axhline(2.5, color="red", linestyle="--", linewidth=0.8, label="2.5 V cutoff")
axes[0].legend()

axes[1].plot(t / 3600, T - 273.15, color="orangered")
axes[1].set_ylabel("Cell temperature / °C")

axes[2].plot(t / 3600, SOC * 100, color="forestgreen")
axes[2].set_ylabel("SOC / %")
axes[2].set_ylim(0, 105)
axes[2].set_xlabel("Time / h")

fig.suptitle("1 C Discharge — SPMe with Lumped Thermal (Chen2020)", fontsize=12)
plt.tight_layout()
plt.show()


## C-Rate Sweep

Higher C-rates cause larger concentration gradients and overpotentials, leading to
steeper voltage drop-off and more pronounced temperature rise.


In [ ]:
results = {}

for c_rate in [0.5, 1.0, 2.0]:
    I_src_r = Constant(c_rate * C_nom)
    T_src_r = Constant(T_amb0)
    cell_r  = CellElectrothermal(initial_soc=1.0)
    sco_r   = Scope(labels=["V", "T", "SOC"])

    sim_r = Simulation(
        blocks=[I_src_r, T_src_r, cell_r, sco_r],
        connections=[
            Connection(I_src_r,       cell_r["I"]),
            Connection(T_src_r,       cell_r["T_amb"]),
            Connection(cell_r["V"],   sco_r[0]),
            Connection(cell_r["T"],   sco_r[1]),
            Connection(cell_r["SOC"], sco_r[2]),
        ],
        dt=10.0,
        Solver=ESDIRK43,
    )

    sim_r.run(1800.0)
    t_r, [V_r, T_r, SOC_r] = sco_r.read()
    results[c_rate] = {"t": t_r, "V": V_r, "T": T_r, "SOC": SOC_r}


In [ ]:
colors = {0.5: "royalblue", 1.0: "forestgreen", 2.0: "orangered"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for c_rate, res in results.items():
    lbl = f"{c_rate} C"
    clr = colors[c_rate]
    axes[0].plot(res["SOC"] * 100, res["V"],                   label=lbl, color=clr)
    axes[1].plot(res["t"] / 3600,  res["T"] - 273.15,          label=lbl, color=clr)

axes[0].set_xlabel("SOC / %")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_xlim(0, 100)
axes[0].set_ylim(2.4, 4.3)
axes[0].axhline(2.5, color="grey", linestyle="--", linewidth=0.8)
axes[0].invert_xaxis()
axes[0].legend()
axes[0].set_title("Voltage vs. SOC")

axes[1].set_xlabel("Time / h")
axes[1].set_ylabel("Cell temperature / °C")
axes[1].legend()
axes[1].set_title("Temperature Rise")

fig.suptitle("C-Rate Comparison — SPMe with Lumped Thermal (Chen2020)", fontsize=12)
plt.tight_layout()
plt.show()


## Summary

- `CellElectrothermal` wraps the PyBaMM SPMe + lumped thermal ODE and integrates it as a standard `DynamicalSystem` in PathSim.
- Higher C-rates produce steeper V–SOC curves and greater temperature rise.
- For an external thermal model (e.g. a custom cooling loop), see notebook 02.
